In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split # 데이터셋 분리 함수
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tensorflow.keras.models import Sequential, save_model, load_model
from tensorflow.keras.layers import Dense, Input, Dropout # 과적합을 줄이기 위한 기법
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

<font size="5" color="black">ch05. 로지스틱 회귀, 분류 (심장병 발병 예측)</font>

```
age: 환자의 나이 (년 단위)
sex: 환자의 성별 (1 = 남성, 0 = 여성)
cp (Chest Pain Type): 흉통의 종류 (0~3 또는 1~4) - 전형적 협심증(Typical angina), 비전형적 협심증(Atypical angina), 비협심증성 통증(Non-anginal pain), 무증상(Asymptomatic)
trestbps: 안정 시 혈압 (입원 시 mm Hg 기준)
chol: 혈청 콜레스테롤 수치 (mg/dl)
fbs: 공복 혈당이 120 mg/dl 초과하는지 여부 (1 = 참, 0 = 거짓)
restecg: 안정 시 심전도 결과 (0 = 정상, 1 = ST-T 파동 이상, 2 = 좌심실 비대 소견)
thalach: 최대 심박수exang: 운동 유발 협심증 여부 (1 = 있음, 0 = 없음)
oldpeak: 안정 시와 비교한 운동으로 인한 ST 분절 하강 수치
slope: 최고 운동 시 ST 분절의 기울기 (상향/평탄/하향)
ca: 형광 투시법으로 확인된 주요 혈관 수 (0~3개)
thal: 지중해빈혈 등 혈액 관련 결함 유형 (정상, 고정 결함, 가역적 결함)
target (class): 심장 질환 진단 결과 (0 = 정상/건강, 1 = 심장병 발병/혈관 협착 50% 초과)
```

# 1. 로지스틱 회귀 분석 (이진분류)
- 데이터 생성 및 전처리
    * 엑셀 - 데이터프레임 - 결측치 처리 - X, y 분리, scale 조정, 데이터셋 분리
- 모델 생성, 학습과정설정, 학습
- 모델 평가 (시각화, 평가, 교차표)
- 모델 사용

## 1.1 데이터 생성 및 전처리

In [9]:
# openpyxl : xlsx 파일을 읽고 쓰는 라이브러리 
%pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.


In [12]:
df = pd.read_excel('data/heart-disease.xlsx')

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   age           303 non-null    int64  
 1   sex           303 non-null    int64  
 2   cp            303 non-null    int64  
 3   treshtbps     303 non-null    int64  
 4   chol          303 non-null    object 
 5   fbs           303 non-null    int64  
 6   restecg       303 non-null    int64  
 7   thalach       303 non-null    int64  
 8   exang         303 non-null    int64  
 9   oldpeak       303 non-null    float64
 10  slope         303 non-null    int64  
 11  ca            303 non-null    object 
 12  hsl           303 non-null    object 
 13  heartDisease  303 non-null    int64  
dtypes: float64(1), int64(10), object(3)
memory usage: 33.3+ KB


In [17]:
df.isnull().sum()

age             0
sex             0
cp              0
treshtbps       0
chol            0
fbs             0
restecg         0
thalach         0
exang           0
oldpeak         0
slope           0
ca              0
hsl             0
heartDisease    0
dtype: int64

In [16]:
df.isin(['?']).sum()

age             0
sex             0
cp              0
treshtbps       0
chol            1
fbs             0
restecg         0
thalach         0
exang           0
oldpeak         0
slope           0
ca              4
hsl             2
heartDisease    0
dtype: int64

In [19]:
df[(df['chol']=='?')|(df['ca']=='?')|(df['hsl']=='?')]

,age,sex,cp,treshtbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,hsl,heartDisease
2,67,1,4,120,?,0,2,129,1,2.6,2,2,7,1
87,53,0,3,128,216,0,2,115,0,0.0,1,0,?,0
166,52,1,3,138,223,0,0,169,0,0.0,1,?,3,0
192,43,1,4,132,247,1,2,143,1,0.1,2,?,7,1
266,52,1,4,128,204,1,0,156,1,1.0,2,0,?,1
287,58,1,2,125,220,0,0,144,0,0.4,2,?,7,0
302,38,1,3,138,175,0,0,173,0,0.0,1,?,3,0


In [23]:
# ?를 결측치로 대체 후 결측치 삭제
df = df.replace('?', np.nan)

In [31]:
# 결측치가 포함된 데이터 추출
df[(df['chol'].isna()|df['ca'].isna()|df['hsl'].isna())]

,age,sex,cp,treshtbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,hsl,heartDisease
2,67,1,4,120,NaN,0,2,129,1,2.6,2,2.0,7.0,1
87,53,0,3,128,216.0,0,2,115,0,0.0,1,0.0,NaN,0
166,52,1,3,138,223.0,0,0,169,0,0.0,1,NaN,3.0,0
192,43,1,4,132,247.0,1,2,143,1,0.1,2,NaN,7.0,1
266,52,1,4,128,204.0,1,0,156,1,1.0,2,0.0,NaN,1
287,58,1,2,125,220.0,0,0,144,0,0.4,2,NaN,7.0,0
302,38,1,3,138,175.0,0,0,173,0,0.0,1,NaN,3.0,0


In [33]:
# 결측치 처리 : dropna(), fillna(), apply()
df = df.dropna(how='any').reset_index(drop=True) # 결측치가 하나의 요소라도 있을 경우, 해당 행 삭제

In [34]:
df.isnull().sum()

age             0
sex             0
cp              0
treshtbps       0
chol            0
fbs             0
restecg         0
thalach         0
exang           0
oldpeak         0
slope           0
ca              0
hsl             0
heartDisease    0
dtype: int64

In [35]:
df.shape

(296, 14)

In [38]:
# 타겟변수 균형 확인
df['heartDisease'].value_counts()/df.shape[0]
df['heartDisease'].value_counts(normalize=True)

0    0.540541
1    0.459459
Name: heartDisease, dtype: float64

In [39]:
# X, y 분리
X = df.iloc[:,:-1].values
y = df.iloc[:,-1:].values
X.shape, y.shape

((296, 13), (296, 1))

In [41]:
# X 변수의 scale 조정

scaler = StandardScaler()

scaled_X = scaler.fit_transform(X)

In [46]:
# 데이터셋 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(scaled_X, y,
                                                    #train_size=0.8 - test_size 지정 시 자동 처리
                                                    test_size=0.2,
                                                    random_state=7, # 난수 시드(Seed)값
                                                    stratify=y # 층화 추출 : 비율을 그대로 유지하면서 데이터를 쪼개는 방식
                                                   )

In [45]:
# 종속 변수 균형 확인
print(pd.DataFrame(y).value_counts(normalize=True))
print(pd.DataFrame(y_train).value_counts(normalize=True))
print(pd.DataFrame(y_test).value_counts(normalize=True))

0    0.540541
1    0.459459
dtype: float64
0    0.542373
1    0.457627
dtype: float64
0    0.533333
1    0.466667
dtype: float64


## 1.2 모델 생성 & 학습과정 설정 & 학습

## 1.3 모델 평가 (시각화, 평가, 교차표)

## 1.4 모델 사용